# Student Pass Predictor (Google Colab)
Exercise 1: Model Persistence · Exercise 2: Flask API · Exercise 3: Streamlit

Run the cells from top to bottom.

In [ ]:
!pip install -q flask scikit-learn pandas joblib streamlit

## Exercise 1 — Model Persistence
Train on study hours + attendance, save as `student_pass_model.joblib`, load it back, and make 3 predictions.

In [ ]:
%%writefile train_model.py
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression

MODEL_FILE = "student_pass_model.joblib"

data = pd.DataFrame({
    "study_hours":           [1, 2, 3, 4, 5, 6, 7, 8, 2, 3, 6, 9, 1, 5, 7, 4],
    "attendance_percentage": [50, 60, 55, 70, 75, 85, 90, 95, 90, 95, 60, 80, 40, 65, 70, 85],
    "passed":                [0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1],
})

X = data[["study_hours", "attendance_percentage"]]
y = data["passed"]

model = LogisticRegression()
model.fit(X, y)

joblib.dump(model, MODEL_FILE)
print(f"Model saved to {MODEL_FILE}")

In [ ]:
!python train_model.py

In [ ]:
import joblib, pandas as pd

loaded = joblib.load("student_pass_model.joblib")   # no retraining here

for hours in (1, 4, 8):
    sample = pd.DataFrame([[hours, 80]], columns=["study_hours", "attendance_percentage"])
    label = "Pass" if loaded.predict(sample)[0] == 1 else "Fail"
    print(f"{hours} study hours, 80% attendance -> {label}")

**Bakit hindi na kailangan i-train ulit?** Ang `fit()` ang nag-aaral ng coefficients at intercept. Sine-save ng `joblib.dump()` ang mga natutunang values na iyon sa file, at ibinabalik ng `joblib.load()` ang parehong fitted model. Kaya ina-apply na lang ng `predict()` ang mga naka-save na values sa bagong input.

## Exercise 2 — Flask API
Validates attendance (0–100) and returns both inputs + prediction as JSON.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
from flask import Flask, jsonify, request

app = Flask(__name__)
model = joblib.load("student_pass_model.joblib")


@app.route("/predict", methods=["GET", "POST"])
def predict():
    source = (request.get_json(silent=True) or {}) if request.method == "POST" else request.args
    try:
        study_hours = float(source.get("study_hours"))
        attendance = float(source.get("attendance_percentage"))
    except (TypeError, ValueError):
        return jsonify(error="study_hours and attendance_percentage must be numbers"), 400

    if study_hours < 0:
        return jsonify(error="study_hours cannot be negative"), 400
    if not 0 <= attendance <= 100:
        return jsonify(error="attendance_percentage must be between 0 and 100"), 400

    X = pd.DataFrame([[study_hours, attendance]],
                     columns=["study_hours", "attendance_percentage"])
    pred = int(model.predict(X)[0])
    prob = float(model.predict_proba(X)[0][1])
    return jsonify(
        study_hours=study_hours,
        attendance_percentage=attendance,
        prediction="Pass" if pred == 1 else "Fail",
        pass_probability=round(prob, 3),
    )

Patakbuhin ang Flask sa background thread, tapos i-test gamit ang `requests` (parang Postman/curl).

In [ ]:
import threading, time, requests
from werkzeug.serving import make_server
from app import app

server = make_server("127.0.0.1", 5000, app)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(1)

url = "http://127.0.0.1:5000/predict"

print("Valid:  ", requests.post(url, json={"study_hours": 5, "attendance_percentage": 85}).json())
print("Invalid:", requests.post(url, json={"study_hours": 5, "attendance_percentage": 150}).json())
print("GET:    ", requests.get(url, params={"study_hours": 2, "attendance_percentage": 40}).json())

In [ ]:
# Optional: same test using curl
!curl -s -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" -d '{"study_hours":6,"attendance_percentage":90}'

## Exercise 3 — Streamlit Interface
Name input, study hours, attendance, Predict button, result message, explanation, and bonus chart.

In [ ]:
%%writefile streamlit_app.py
import joblib
import pandas as pd
import streamlit as st

model = joblib.load("student_pass_model.joblib")

st.title("Student Pass Predictor")

name = st.text_input("Student name")
hours = st.number_input("Study hours per day", min_value=0.0, max_value=24.0, value=3.0, step=0.5)
attendance = st.slider("Attendance percentage", 0, 100, 80)

if st.button("Predict"):
    X = pd.DataFrame([[hours, attendance]], columns=["study_hours", "attendance_percentage"])
    pred = int(model.predict(X)[0])
    prob = float(model.predict_proba(X)[0][1])
    who = name.strip() or "This student"

    if pred == 1:
        st.success(f"{who} is predicted to PASS ({prob:.0%} pass probability).")
    else:
        st.error(f"{who} is predicted to FAIL ({prob:.0%} pass probability).")

    st.write(
        "**How this was generated:** a logistic regression model, trained earlier on past "
        "study hours and attendance, was loaded from `student_pass_model.joblib`. It combines "
        "the two inputs into a pass probability; 50% or higher is classified as Pass."
    )

    sweep = pd.DataFrame({"study_hours": [h / 2 for h in range(0, 21)]})
    sweep["attendance_percentage"] = attendance
    sweep["predicted_class"] = model.predict(sweep[["study_hours", "attendance_percentage"]])
    st.subheader("Study hours vs predicted class (1 = Pass, 0 = Fail)")
    st.line_chart(sweep.set_index("study_hours")["predicted_class"])

Hindi direktang nabubuksan ang Streamlit sa Colab, kaya gagamit tayo ng free Cloudflare tunnel. Hintayin ang link na `https://....trycloudflare.com` at i-click ito.

In [ ]:
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port", "8501",
                  "--server.headless", "true"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Open this link:", m.group(0))
        break